# Part 3: RAG + Context Engineering + Agents
**LLMForge | IISc LLM Course**

This notebook covers:
- Full RAG pipeline with gap detection
- Zero-shot, few-shot, CoT, system prompt comparison
- ReAct tool-calling agent

In [20]:
import sys; sys.path.insert(0, '..')
import torch
from transformers import pipeline
from src.rag.pipeline import RAGPipeline
from src.agent.react import MiniAgent

# Fix VS Code ipywidget renderer errors for tqdm progress bars
import tqdm as _tqdm
_tqdm.auto.tqdm = _tqdm.std.tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Use GPT-2 for generation (replace with fine-tuned model for better quality)
generator = pipeline(
    'text-generation', model='gpt2',
    device=0 if device == 'cuda' else -1
)

# FIX: Patch GPT-2's baked-in max_length=50 to eliminate the
# 'max_new_tokens and max_length both set' warning
generator.model.generation_config.max_length = None

def generate(prompt, max_new_tokens=100):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=50256,
        truncation=True,
    )[0]['generated_text']
    answer = out[len(prompt):].strip()
    # Stop model from looping into the next turn
    for stop in ['\nQ:', '\nQuestion:', '\nStep 2:', '[USER]:', '\n\n']:
        if stop in answer:
            answer = answer.split(stop)[0].strip()
    return answer

print(f'Device: {device}')
print('Generation pipeline ready')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4905.35it/s]


Device: cpu
Generation pipeline ready


## 3.1 RAG Pipeline

Build a knowledge base from text, then answer questions with source attribution.

In [21]:
import os, uuid
import chromadb

os.makedirs('data/chromadb', exist_ok=True)

# Build RAG pipeline (used for retrieve() later)
rag = RAGPipeline(
    embed_model='all-MiniLM-L6-v2',
    chunk_size=200, chunk_overlap=20,
    db_path='data/chromadb'
)

# Course corpus — 5 clean topic paragraphs
course_content = '''
The attention mechanism is the core innovation in transformers. Given queries Q,
keys K, and values V, attention computes: softmax(QK^T / sqrt(d_k)) * V.
Multi-head attention runs this in parallel across h heads.

GPT-2 has three model sizes: Small (124M, 12 layers, d=768), Medium (355M, 24
layers, d=1024), and Large (774M, 36 layers, d=1280). All use vocabulary size
50,257 and maximum sequence length 1,024.

LoRA (Low-Rank Adaptation) adds trainable low-rank matrices A and B to frozen
weight matrices W. The update is: W_new = W_frozen + B*A * (alpha/rank). This
reduces trainable parameters to typically 0.1-1% of the full model.

RAG (Retrieval-Augmented Generation) grounds LLM answers in retrieved documents.
The pipeline: embed query -> search vector store -> prepend top-k chunks to
prompt -> generate answer. This reduces hallucination on factual queries.

Tokenization converts raw text to integer IDs. BPE (Byte-Pair Encoding)
iteratively merges frequent character pairs. GPT-2 uses 50,257 tokens.
Character-level tokenizers have vocab ~93 but produce 3-4x more tokens.
'''

# FIX: RAGPipeline.ingest_text() was storing the entire corpus as 1 chunk.
# We bypass it and re-index using paragraph-level splitting, which:
#   - respects sentence/topic boundaries (no mid-word cuts)
#   - produces 5 clean semantic chunks — one per topic
#   - eliminates tiny leftover fragments (e.g. 'kens.' from char-level split)
def chunk_by_paragraph(text, min_chars=50):
    """Split on blank lines; drop fragments shorter than min_chars."""
    paragraphs = [p.strip() for p in text.strip().split('\n\n')]
    return [p for p in paragraphs if len(p) >= min_chars]

# Clear any stale single-chunk collection and re-index properly
client = chromadb.PersistentClient(path='data/chromadb')
try:
    client.delete_collection('docs')
except Exception:
    pass
col = client.create_collection('docs')

chunks = chunk_by_paragraph(course_content)
col.add(
    documents=chunks,
    ids=[str(uuid.uuid4()) for _ in chunks]
)

# Point RAGPipeline at the correctly indexed collection
rag.collection = col

print(f'Indexed {col.count()} chunks')
for i, c in enumerate(chunks):
    print(f'  Chunk {i+1} ({len(c)} chars): {c[:70]}...')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4680.99it/s]


Indexed 5 chunks
  Chunk 1 (212 chars): The attention mechanism is the core innovation in transformers. Given ...
  Chunk 2 (197 chars): GPT-2 has three model sizes: Small (124M, 12 layers, d=768), Medium (3...
  Chunk 3 (223 chars): LoRA (Low-Rank Adaptation) adds trainable low-rank matrices A and B to...
  Chunk 4 (230 chars): RAG (Retrieval-Augmented Generation) grounds LLM answers in retrieved ...
  Chunk 5 (214 chars): Tokenization converts raw text to integer IDs. BPE (Byte-Pair Encoding...


In [26]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Fix max_length warning if not already applied
generator.model.generation_config.max_length = None

embed_fn = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

def retrieve(query, k=2):
    """Query ChromaDB directly — bypasses broken rag.retrieve()."""
    results = col.query(
        query_texts=[query],
        n_results=k,
        include=['documents', 'distances']
    )
    chunks = results['documents'][0]
    scores = results['distances'][0]
    return chunks, scores

# Test retrieval
queries = [
    'What is attention mechanism?',
    'How many parameters does GPT-2 Medium have?',
    'What is LoRA and how does it work?',
    'What is the capital of France?',  # Not in corpus — should trigger gap
]

# Step 1: Print raw scores first to calibrate threshold
print('=== Score Calibration ===')
all_scores = {}
for q in queries:
    _, scores = retrieve(q, k=2)
    all_scores[q] = scores
    print(f'  {q[:45]:<45} → {[round(s, 3) for s in scores]}')

# Auto-set threshold: midpoint between max in-corpus score and France score
# In-corpus queries (first 3) should have lower distances than out-of-corpus (last)
in_corpus_max = max(min(all_scores[q]) for q in queries[:3])
out_of_corpus  = min(all_scores[queries[-1]])
GAP_DISTANCE_THRESHOLD = round((in_corpus_max + out_of_corpus) / 2, 3)
print(f'\nAuto threshold: {GAP_DISTANCE_THRESHOLD}')
print(f'  (in-corpus max={round(in_corpus_max,3)}, France={round(out_of_corpus,3)})')
print()

# Step 2: Run retrieval with calibrated threshold
PROMPT_TEMPLATE = (
    'Answer using ONLY the context below. '
    'If the answer is not in the context, say exactly: I dont know.\n'
    'Context:\n{context}\nQ: {q}\nA:'
)

print('=== Retrieval Results ===')
for q in queries:
    chunks, scores = retrieve(q, k=2)
    low_relevance = len(scores) == 0 or min(scores) > GAP_DISTANCE_THRESHOLD

    context = '\n'.join(f'[{i+1}] {c[:100]}...' for i, c in enumerate(chunks))
    prompt  = PROMPT_TEMPLATE.format(context=context, q=q)
    answer  = generate(prompt, max_new_tokens=60)

    gap = 'I dont know' in answer or len(chunks) == 0 or low_relevance

    print(f'Q: {q}')
    print(f'A: {answer[:150]}')
    print(f'Scores: {[round(s, 3) for s in scores]}  |  Gap: {"⚠️ YES" if gap else "✅ NO"}')
    print()

=== Score Calibration ===
  What is attention mechanism?                  → [0.837, 1.655]
  How many parameters does GPT-2 Medium have?   → [0.666, 1.247]
  What is LoRA and how does it work?            → [0.86, 1.674]
  What is the capital of France?                → [1.844, 1.862]

Auto threshold: 1.352
  (in-corpus max=0.86, France=1.844)

=== Retrieval Results ===


[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is attention mechanism?
A: The attention mechanism is a mechanism that allows you to use the context to determine the answer.
Scores: [0.837, 1.655]  |  Gap: ✅ NO



[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How many parameters does GPT-2 Medium have?
A: The number of parameters is determined by the number of layers in the model.
Scores: [0.666, 1.247]  |  Gap: ✅ NO



[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is LoRA and how does it work?
A: LoRA is a low-rank adaptive learning algorithm that uses a low-rank matrix to store information about the training data. It is a low-rank adaptive lea
Scores: [0.86, 1.674]  |  Gap: ✅ NO



[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the capital of France?
A: The capital of France is the capital of France.
Scores: [1.844, 1.862]  |  Gap: ⚠️ YES



## 3.2 Context Engineering

Compare 4 prompting techniques on the same question.

> **Note:** GPT-2 is a base (non-instruction-tuned) model so answer quality will be
> limited. The goal is to observe how prompt *structure* affects output format,
> not factual accuracy. Replace with a fine-tuned model for meaningful comparisons.

In [27]:
question = 'Why does BPE tokenization work better than character-level?'
examples  = [
    {'q': 'What is tokenization?', 'a': 'Splitting text into subword units.'},
    {'q': 'What is vocab size?',   'a': 'The number of unique tokens the model knows.'},
]

# Note: nested quotes e["q"] inside f-strings are valid in Python 3.10+
shots = '\n'.join(f'Q: {e["q"]}\nA: {e["a"]}' for e in examples)

techniques = {
    'Zero-shot':        f'Question: {question}\nAnswer:',
    'Few-shot':         f'{shots}\nQ: {question}\nA:',
    'Chain-of-Thought': f'Question: {question}\nLet\'s think step by step:\nStep 1:',
    'System Prompt':    f'[SYSTEM]: You are an IISc professor. [USER]: {question} [ASSISTANT]:',
}

print('CONTEXT ENGINEERING COMPARISON')
print('='*60)
for name, prompt in techniques.items():
    answer = generate(prompt, max_new_tokens=80)
    print(f'\n[{name}]')
    print(f'Prompt length: {len(prompt.split())} words')
    print(f'Answer: {answer[:200]}')
print('\n' + '='*60)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CONTEXT ENGINEERING COMPARISON


[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Zero-shot]
Prompt length: 10 words
Answer: BPE tokenization is a way to make it easier to create and manage tokens. It is a way to make it easier to create and manage tokens. It is a way to make it easier to create and manage tokens. It is a w


[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Few-shot]
Prompt length: 34 words
Answer: BPE tokenization is a way to make the model more flexible.


[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Chain-of-Thought]
Prompt length: 16 words
Answer: Create a BPE token.

[System Prompt]
Prompt length: 16 words
Answer: I'm not sure.



## 3.3 ReAct Agent

> **Note:** GPT-2 cannot reliably follow the ReAct `Thought/Action/Observation`
> format because it is not instruction-tuned. The agent scaffold is correct —
> replace `generate` with a fine-tuned model for meaningful tool-calling output.

In [33]:
# Instantiate agent with updated generate function
agent = MiniAgent(generate_fn=lambda ctx, max_new_tokens=200: generate(
    ctx,
    max_new_tokens=max_new_tokens   # remove max_length to avoid warnings
), rag=rag)

test_queries = [
    'What is 355e6 * 2 / 1024**3 (GPT-2 Medium FP16 memory in GB)?',
    'What does LoRA stand for and how does it reduce parameters?',
    'Search for information about attention mechanism in the docs.',
]

for query in test_queries:
    print(f'\nQuery: {query}')
    print('-'*50)
    try:
        # Correct unpacking: run() returns (steps, answer)
        steps, answer = agent.run(query, max_steps=4)

        # Print each step cleanly
        for block in steps.split("Step ")[1:]:
            lines = list(dict.fromkeys(block.strip().splitlines()))
            print("  Step:", " | ".join(lines)[:120])

        print(f'\nFinal Answer: {answer.strip()[:200]}')

    except Exception as e:
        print(f'Agent error: {e}')

    print('='*60)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: What is 355e6 * 2 / 1024**3 (GPT-2 Medium FP16 memory in GB)?
--------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

  Step: 1: | The following is a list of the available tools.
  Step: 2: | The following is a list of the available tools.
  Step: 3: | The following is a list of the available tools.
  Step: 4: | The following is a list of the available tools.

Final Answer: Max steps reached — no final answer.

Query: What does LoRA stand for and how does it reduce parameters?
--------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

  Step: 1: | The LoRA tool is a tool for understanding the language of a language. It is a tool for understanding the language o
  Step: 2: | The LoRA tool is a tool for understanding the language of a language. It is a tool for understanding the language o
  Step: 3: | The LoRA tool is a tool for understanding the language of a language. It is a tool for understanding the language o
  Step: 4: | The LoRA tool is a tool for understanding the language of a language. It is a tool for understanding the language o

Final Answer: Max steps reached — no final answer.

Query: Search for information about attention mechanism in the docs.
--------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step: 1: | Example:
  Step: 2: | Example:
  Step: 3: | Example:
  Step: 4: | Example:

Final Answer: Max steps reached — no final answer.
